#TF IDF + Logistic Regression

##Topik 0

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from imblearn import over_sampling
from IPython.display import display

# Reading product review sentiment file
df_topik0 = pd.read_csv('topik0.csv',\
                      encoding='latin-1')
display(df_topik0.sample(n=5, random_state=42))

,text,label_st1,sentiment
4365,fantastic product fantastic product leave room...,0,1
1129,amaze love game love movie may little still aw...,0,1
2602,good childrens movie get daughter love theater...,0,1
1747,love smell love smell usually really enjoy aus...,0,1
3286,dat roar tho saw movie time still love cant ...,0,1


In [ ]:
df_topik0.shape

(6025, 3)

In [ ]:
x=df_topik0['text']
y=df_topik0['sentiment']
print("Distribusi sentimen positif dan negatif dalam ulasan: \n{}".format(y.value_counts(normalize=True)))
# Split the dataset into test and train
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=50)

# As we saw above that data is imbalanced, balance training data using over sampling

ros = over_sampling.RandomOverSampler(random_state=0)
X_train, y_train = ros.fit_resample(pd.DataFrame(X_train), pd.Series(y_train))
print("Distribusi sentimen positif dan negatif dalam ulasan setelah oversampling: \n{}".format(y_train.value_counts(normalize=True)))
# Convert into list of string
X_train = X_train['text'].tolist()

Distribusi sentimen positif dan negatif dalam ulasan: 
sentiment
1    0.875353
0    0.124647
Name: proportion, dtype: float64
Distribusi sentimen positif dan negatif dalam ulasan setelah oversampling: 
sentiment
1    0.5
0    0.5
Name: proportion, dtype: float64


In [ ]:
word_vectorizer = TfidfVectorizer(strip_accents='unicode', token_pattern=r'\w{1,}',\
                                ngram_range=(1, 3), stop_words='english', sublinear_tf=True)

# Fiting it on Train
word_vectorizer.fit(X_train)
# Transforming the train and test datasets
X_train_transformed = word_vectorizer.transform(X_train)
X_test_transformed = word_vectorizer.transform(X_test.tolist())

In [ ]:
def evaluate_model(y_pred,y_actual):
    print(classification_report(y_true = y_actual, y_pred = y_pred))
    # Confusion matrix
    cm = confusion_matrix(y_true = y_actual, y_pred = y_pred)
    TN = cm[0, 0]
    FP = cm[0, 1]
    FN = cm[1, 0]
    TP = cm[1, 1]
    # Calculating the Sensitivity
    sensitivity = round(TP/float(FN + TP),2)
    print("sensitivity: {}".format(sensitivity))
    # Calculating the Specificity
    specificity = round(TN / float(TN + FP),2)
    print("specificity: {}".format(specificity))

#4.1 Model Training
logit = LogisticRegression()
logit.fit(X_train_transformed,y_train)
#4.2 Prediction on Train Data
y_pred_train= logit.predict(X_train_transformed)
#4.3 Prediction on Test Data
y_pred_test = logit.predict(X_test_transformed)
#4.4 Evaluation on Train
print("Evaluation on Train dataset ..")
evaluate_model(y_pred = y_pred_train, y_actual = y_train)
print("Evaluation on Test dataset ..")
#4.5 Evaluation on Test
evaluate_model(y_pred = y_pred_test, y_actual = y_test)

Evaluation on Train dataset ..
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      3696
           1       1.00      0.99      1.00      3696

    accuracy                           1.00      7392
   macro avg       1.00      1.00      1.00      7392
weighted avg       1.00      1.00      1.00      7392

sensitivity: 0.99
specificity: 1.0
Evaluation on Test dataset ..
              precision    recall  f1-score   support

           0       0.47      0.46      0.46       230
           1       0.92      0.92      0.92      1578

    accuracy                           0.87      1808
   macro avg       0.69      0.69      0.69      1808
weighted avg       0.86      0.87      0.86      1808

sensitivity: 0.92
specificity: 0.46


##Error Analysis

In [ ]:
true_labels = y_test.tolist()
pred_labels = y_pred_test.tolist()
texts = X_test.tolist()

error_df0 = pd.DataFrame({
    'text': texts,
    'actual': true_labels,
    'predicted': pred_labels
})

In [ ]:
# Inspect misclassified samples
misclassified_0 = error_df0[error_df0['actual'] != error_df0['predicted']]

print("Jumlah total error:", len(misclassified_0))
display(misclassified_0.sample(10, random_state=42))

Jumlah total error: 244


,text,actual,predicted
157,worth hype receive product influenster exchang...,0,1
35,right track product allow keep pending item it...,0,1
1045,great styling gel would definitely recommend,0,1
1497,son love movie airplane always entertain go ma...,1,0
1393,rust bust bought month ago rust pretty quickly...,1,0
1206,kid movie kid love kept four year old quiet ad...,1,0
1341,great set movie previous resident evil movie e...,1,0
795,loyal customer dont use anything else turn ...,0,1
47,work wonder product totally eliminated frizz p...,1,0
673,awesome b grear,0,1


###Sample df0

In [ ]:
sampled_df0 = error_df0.sample(n=300, random_state=42)
sampled_df0

,text,actual,predicted
1613,great workout fan leslies several year year...,1,1
65,fabu vacuum quite bit research hepa vacuum set...,0,1
700,good smell effective like smell effectiveness ...,0,1
938,great juicer previously purchase diferent juic...,1,1
1623,handle broken twice like vacuum huge exception...,0,0
...,...,...,...
71,easy apply greasy overpower smell complaint sp...,1,1
1173,streak film product near work successfully cle...,1,0
259,nice answer look like nice answer organize pre...,1,1
51,awesome great product would recommend product ...,1,1


In [ ]:
# Inspect misclassified samples
misclassified_sample0 = sampled_df0[sampled_df0['actual'] != sampled_df0['predicted']]

print("Jumlah total error:", len(misclassified_sample0))
display(misclassified_sample0.sample(10, random_state=42))

misclassified_0.to_csv('misclassified0.csv', index=False)

Jumlah total error: 36


,text,actual,predicted
1173,streak film product near work successfully cle...,1,0
1487,like new gel finally find perfect gel thick ha...,1,0
324,bring back coco bad business driven somewhere ...,0,1
1273,windex change use windex forever last couple b...,1,0
212,miss color love burt bee lipshimmers time ...,1,0
767,disappointeed one bad movie make im happy purc...,0,1
1393,rust bust bought month ago rust pretty quickly...,1,0
275,can not buy good bisquit mix use bisquick sinc...,1,0
303,waste money purchase think would sort warm eff...,1,0
210,give baby rash us blister like bump hand foot ...,1,0


In [ ]:
sampled_with_acc_0 = sampled_df0.copy()

# Add column acc_b to the copy
sampled_with_acc_0['acc_b'] = sampled_with_acc_0['text'].apply(
    lambda x: 0 if x in misclassified_sample0['text'].values else 1
)

# Show result
display(sampled_with_acc_0)

,text,actual,predicted,acc_b
1613,great workout fan leslies several year year...,1,1,1
65,fabu vacuum quite bit research hepa vacuum set...,0,1,0
700,good smell effective like smell effectiveness ...,0,1,0
938,great juicer previously purchase diferent juic...,1,1,1
1623,handle broken twice like vacuum huge exception...,0,0,1
...,...,...,...,...
71,easy apply greasy overpower smell complaint sp...,1,1,1
1173,streak film product near work successfully cle...,1,0,0
259,nice answer look like nice answer organize pre...,1,1,1
51,awesome great product would recommend product ...,1,1,1


In [ ]:
sampled_with_acc_0.to_csv('topik0_sampel.csv', index=False)

##Topik 1

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from imblearn import over_sampling
from IPython.display import display

# Reading product review sentiment file
df_topik1 = pd.read_csv('topik1.csv',\
                      encoding='latin-1')
display(df_topik1.sample(n=5, random_state=42))

,text,label_st1,sentiment
5653,clorox disinfect wipe awesome use clorox disin...,1,1
6978,great movie grand child grand child watch movi...,1,1
623,clorox wipe r awesome love clorox wipe conveni...,1,1
8220,awesome wipe love easy acessible canister dont...,1,1
8275,get job do really clean disinfects need clean ...,1,1


In [ ]:
df_topik1.shape

(9112, 3)

In [ ]:
x=df_topik1['text']
y=df_topik1['sentiment']
print("Distribusi sentimen positif dan negatif dalam ulasan: \n{}".format(y.value_counts(normalize=True)))
# Split the dataset into test and train
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=50)

# As we saw above that data is imbalanced, balance training data using over sampling

ros = over_sampling.RandomOverSampler(random_state=0)
X_train, y_train = ros.fit_resample(pd.DataFrame(X_train), pd.Series(y_train))
print("Distribusi sentimen positif dan negatif dalam ulasan setelah oversampling: \n{}".format(y_train.value_counts(normalize=True)))
# Convert into list of string
X_train = X_train['text'].tolist()

Distribusi sentimen positif dan negatif dalam ulasan: 
sentiment
1    0.897717
0    0.102283
Name: proportion, dtype: float64
Distribusi sentimen positif dan negatif dalam ulasan setelah oversampling: 
sentiment
1    0.5
0    0.5
Name: proportion, dtype: float64


In [ ]:
word_vectorizer = TfidfVectorizer(strip_accents='unicode', token_pattern=r'\w{1,}',\
                                ngram_range=(1, 3), stop_words='english', sublinear_tf=True)

# Fiting it on Train
word_vectorizer.fit(X_train)
# Transforming the train and test datasets
X_train_transformed = word_vectorizer.transform(X_train)
X_test_transformed = word_vectorizer.transform(X_test.tolist())

In [ ]:
def evaluate_model(y_pred,y_actual):
    print(classification_report(y_true = y_actual, y_pred = y_pred))
    # Confusion matrix
    cm = confusion_matrix(y_true = y_actual, y_pred = y_pred)
    TN = cm[0, 0]
    FP = cm[0, 1]
    FN = cm[1, 0]
    TP = cm[1, 1]
    # Calculating the Sensitivity
    sensitivity = round(TP/float(FN + TP),2)
    print("sensitivity: {}".format(sensitivity))
    # Calculating the Specificity
    specificity = round(TN / float(TN + FP),2)
    print("specificity: {}".format(specificity))

#4.1 Model Training
logit = LogisticRegression()
logit.fit(X_train_transformed,y_train)
#4.2 Prediction on Train Data
y_pred_train= logit.predict(X_train_transformed)
#4.3 Prediction on Test Data
y_pred_test = logit.predict(X_test_transformed)
#4.4 Evaluation on Train
print("Evaluation on Train dataset ..")
evaluate_model(y_pred = y_pred_train, y_actual = y_train)
print("Evaluation on Test dataset ..")
#4.5 Evaluation on Test
evaluate_model(y_pred = y_pred_test, y_actual = y_test)

Evaluation on Train dataset ..
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      5745
           1       1.00      0.99      0.99      5745

    accuracy                           0.99     11490
   macro avg       0.99      0.99      0.99     11490
weighted avg       0.99      0.99      0.99     11490

sensitivity: 0.99
specificity: 1.0
Evaluation on Test dataset ..
              precision    recall  f1-score   support

           0       0.51      0.49      0.50       299
           1       0.94      0.94      0.94      2435

    accuracy                           0.89      2734
   macro avg       0.72      0.72      0.72      2734
weighted avg       0.89      0.89      0.89      2734

sensitivity: 0.94
specificity: 0.49


###Error Analysis

In [ ]:
true_labels = y_test.tolist()
pred_labels = y_pred_test.tolist()
texts = X_test.tolist()

error_df1 = pd.DataFrame({
    'text': texts,
    'actual': true_labels,
    'predicted': pred_labels
})

In [ ]:
# Inspect misclassified samples
misclassified_1 = error_df1[error_df1['actual'] != error_df1['predicted']]

print("Jumlah total error:", len(misclassified_1))
display(misclassified_1.sample(10, random_state=42))

Jumlah total error: 293


,text,actual,predicted
835,product clean disinfect great like wipe everyt...,0,1
2453,best room heater ever late husband find great ...,1,0
420,love smell love smell tghe disinfect bathroom ...,0,1
1627,favorite hand weve use year toddler toys fight...,1,0
1364,love product use home well vehicle,0,1
1206,work great clorox wipe household staple use ne...,0,1
2126,vicks vapor rub vicks rub effective cold flu s...,1,0
1138,go little packet great sit questionable picnic...,0,1
79,disinfects stuff really works clean get stain ...,1,0
918,dog people godsend new dog niece life make eno...,1,0


###Sample df1

In [ ]:
sampled_df1 = error_df1.sample(n=300, random_state=42)
sampled_df1

,text,actual,predicted
1001,wipe clean disinfect pleasant scent clorox wip...,1,1
1837,great gift grand kid love highly recommend,1,1
2480,excellent product love clorox product clean wh...,1,1
942,great product great product easy use keep ever...,1,1
73,love wipe love clorox wipe use clean need revi...,1,1
...,...,...,...
415,smell great love disinfect power clorox wipe a...,1,1
2691,excellent bug killer favorite product use espe...,1,0
1270,clean love esp new scent review collect part p...,1,1
1253,wouldnt know use clorox wipe everything always...,1,1


In [ ]:
# Inspect misclassified samples
misclassified_sample1 = sampled_df1[sampled_df1['actual'] != sampled_df1['predicted']]

print("Jumlah total error:", len(misclassified_sample1))
display(misclassified_sample1.sample(10, random_state=42))

misclassified_1.to_csv('misclassified1.csv', index=False)

Jumlah total error: 25


,text,actual,predicted
1298,kill germ love clorox product especially flu s...,1,0
354,love wipe good use everything never disappoint...,0,1
485,trap good open bag cant stop eating,0,1
173,well builtsturdyattractive functional go perfe...,0,1
2261,clorox wipe product must family also work clor...,1,0
1614,still staple clean supply cant remenber time w...,1,0
1360,classroom enjoy use clorox wipe classroom help...,1,0
2486,disinfect wipe excellent supperduper product t...,1,0
1960,kill viruses one swipe main reason use clorox ...,1,0
785,delicious tasty kid love may leave finger litt...,1,0


In [ ]:
sampled_with_acc_1 = sampled_df1.copy()
sampled_with_acc_1['acc_c'] = sampled_with_acc_1['text'].apply(
    lambda x: 0 if x in misclassified_sample1['text'].values else 1
)
display(sampled_with_acc_1)

,text,actual,predicted,acc_c
1001,wipe clean disinfect pleasant scent clorox wip...,1,1,1
1837,great gift grand kid love highly recommend,1,1,1
2480,excellent product love clorox product clean wh...,1,1,1
942,great product great product easy use keep ever...,1,1,1
73,love wipe love clorox wipe use clean need revi...,1,1,1
...,...,...,...,...
415,smell great love disinfect power clorox wipe a...,1,1,1
2691,excellent bug killer favorite product use espe...,1,0,0
1270,clean love esp new scent review collect part p...,1,1,1
1253,wouldnt know use clorox wipe everything always...,1,1,1


In [ ]:
sampled_with_acc_1.to_csv('topik1_sampel.csv', index=False)

##Topik 2

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from imblearn import over_sampling
from IPython.display import display

# Reading product review sentiment file
df_topik2 = pd.read_csv('topik2.csv',\
                      encoding='latin-1')
display(df_topik2.sample(n=5, random_state=42))

,text,label_st1,sentiment
324,great movie series watch love resident evil mo...,2,0
5395,great idea really liked concept folder wish se...,2,1
6084,excellent product ready index double column ta...,2,1
6323,incredible remake film shot humorously lots fa...,2,1
381,great sequel funny scene wife love first get s...,2,1


In [ ]:
df_topik2.shape

(6424, 3)

In [ ]:
x=df_topik2['text']
y=df_topik2['sentiment']
print("Distribusi sentimen positif dan negatif dalam ulasan: \n{}".format(y.value_counts(normalize=True)))
# Split the dataset into test and train
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=50)

# As we saw above that data is imbalanced, balance training data using over sampling

ros = over_sampling.RandomOverSampler(random_state=0)
X_train, y_train = ros.fit_resample(pd.DataFrame(X_train), pd.Series(y_train))
print("Distribusi sentimen positif dan negatif dalam ulasan setelah oversampling: \n{}".format(y_train.value_counts(normalize=True)))
# Convert into list of string
X_train = X_train['text'].tolist()

Distribusi sentimen positif dan negatif dalam ulasan: 
sentiment
1    0.911426
0    0.088574
Name: proportion, dtype: float64
Distribusi sentimen positif dan negatif dalam ulasan setelah oversampling: 
sentiment
1    0.5
0    0.5
Name: proportion, dtype: float64


In [ ]:
word_vectorizer = TfidfVectorizer(strip_accents='unicode', token_pattern=r'\w{1,}',\
                                ngram_range=(1, 3), stop_words='english', sublinear_tf=True)

# Fiting it on Train
word_vectorizer.fit(X_train)
# Transforming the train and test datasets
X_train_transformed = word_vectorizer.transform(X_train)
X_test_transformed = word_vectorizer.transform(X_test.tolist())

In [ ]:
def evaluate_model(y_pred,y_actual):
    print(classification_report(y_true = y_actual, y_pred = y_pred))
    # Confusion matrix
    cm = confusion_matrix(y_true = y_actual, y_pred = y_pred)
    TN = cm[0, 0]
    FP = cm[0, 1]
    FN = cm[1, 0]
    TP = cm[1, 1]
    # Calculating the Sensitivity
    sensitivity = round(TP/float(FN + TP),2)
    print("sensitivity: {}".format(sensitivity))
    # Calculating the Specificity
    specificity = round(TN / float(TN + FP),2)
    print("specificity: {}".format(specificity))

#4.1 Model Training
logit = LogisticRegression()
logit.fit(X_train_transformed,y_train)
#4.2 Prediction on Train Data
y_pred_train= logit.predict(X_train_transformed)
#4.3 Prediction on Test Data
y_pred_test = logit.predict(X_test_transformed)
#4.4 Evaluation on Train
print("Evaluation on Train dataset ..")
evaluate_model(y_pred = y_pred_train, y_actual = y_train)
print("Evaluation on Test dataset ..")
#4.5 Evaluation on Test
evaluate_model(y_pred = y_pred_test, y_actual = y_test)

Evaluation on Train dataset ..
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      4095
           1       1.00      0.99      1.00      4095

    accuracy                           1.00      8190
   macro avg       1.00      1.00      1.00      8190
weighted avg       1.00      1.00      1.00      8190

sensitivity: 0.99
specificity: 1.0
Evaluation on Test dataset ..
              precision    recall  f1-score   support

           0       0.41      0.35      0.38       168
           1       0.94      0.95      0.95      1760

    accuracy                           0.90      1928
   macro avg       0.67      0.65      0.66      1928
weighted avg       0.89      0.90      0.90      1928

sensitivity: 0.95
specificity: 0.35


###Error Analysis

In [ ]:
true_labels = y_test.tolist()
pred_labels = y_pred_test.tolist()
texts = X_test.tolist()

error_df2 = pd.DataFrame({
    'text': texts,
    'actual': true_labels,
    'predicted': pred_labels
})

In [ ]:
# Inspect misclassified samples
misclassified_2 = error_df2[error_df2['actual'] != error_df2['predicted']]

print("Jumlah total error:", len(misclassified_2))
display(misclassified_2.sample(10, random_state=42))

Jumlah total error: 194


,text,actual,predicted
1274,hilarious cry laugh rude crude humorbut make m...,0,1
206,funny funny first one watch movie three time a...,1,0
1451,goood movie good movie son like watch movie li...,1,0
862,genius help stay organised different counselin...,0,1
625,awesome cd would recommend friend family member,0,1
1408,efficient help stay organize used organize bil...,0,1
553,excellent collection collection amazing must b...,1,0
187,love love jason aldean disappoint purchasing cd,0,1
1009,every room im huge clorox fan wipe quick convient,1,0
991,great movie plane lover get gift plane lover f...,0,1


###Sample df2

In [ ]:
sampled_df2 = error_df2.sample(n=300, random_state=42)
sampled_df2

,text,actual,predicted
1118,great entertainment quite like classic japanes...,1,1
1652,king genre definitely best godzilla remake hom...,1,1
1863,great yearly report use office keep track acco...,1,1
941,great comeback classic movie good job pull aud...,1,1
331,dark shadow love dark humor movie eva green be...,1,0
...,...,...,...
591,funny movie fan cast movie push limit little b...,1,1
1160,refresh taste love product hard time find stor...,0,1
1189,godzilla godzilla movie fan like movie,0,1
1538,big guy back one best reboots movie series eve...,1,1


In [ ]:
# Inspect misclassified samples
misclassified_sample2 = sampled_df2[sampled_df2['actual'] != sampled_df2['predicted']]

print("Jumlah total error:", len(misclassified_sample2))
display(misclassified_sample2.sample(10, random_state=42))

misclassified_2.to_csv('misclassified2.csv', index=False)

Jumlah total error: 30


,text,actual,predicted
1523,best godzilla way dramatic godzilla moviehe se...,0,1
1817,tim burton creative guiness see every tim burt...,0,1
394,ketchup awesome consider ketchup snob ill pas ...,0,1
879,godzilla back complaint fact movie focus bit m...,0,1
1009,every room im huge clorox fan wipe quick convient,1,0
744,good movie like movie wait long bring godzilla...,1,0
1160,refresh taste love product hard time find stor...,0,1
162,arrid formula work body chemistry take local s...,0,1
1890,true fan resident evil series resident evil mo...,1,0
331,dark shadow love dark humor movie eva green be...,1,0


In [ ]:
sampled_with_acc_2 = sampled_df2.copy()
sampled_with_acc_2['acc_d'] = sampled_with_acc_2['text'].apply(
    lambda x: 0 if x in misclassified_sample2['text'].values else 1
)
display(sampled_with_acc_2)

,text,actual,predicted,acc_d
1118,great entertainment quite like classic japanes...,1,1,1
1652,king genre definitely best godzilla remake hom...,1,1,1
1863,great yearly report use office keep track acco...,1,1,1
941,great comeback classic movie good job pull aud...,1,1,1
331,dark shadow love dark humor movie eva green be...,1,0,0
...,...,...,...,...
591,funny movie fan cast movie push limit little b...,1,1,1
1160,refresh taste love product hard time find stor...,0,1,0
1189,godzilla godzilla movie fan like movie,0,1,0
1538,big guy back one best reboots movie series eve...,1,1,1


In [ ]:
sampled_with_acc_2.to_csv('topik2_sampel.csv', index=False)

##Topik 3

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from imblearn import over_sampling
from IPython.display import display

# Reading product review sentiment file
df_topik3 = pd.read_csv('topik3.csv',\
                      encoding='latin-1')
display(df_topik3.sample(n=5, random_state=42))

,text,label_st1,sentiment
1025,great remake great movie action pack always en...,3,1
4006,pretty good movie good kid adult good sequel ...,3,1
2594,entertain like plane enjoy fire rescue good fo...,3,1
1817,bounce use bounce year love always use need tr...,3,1
1620,stuff amaze lip always problem get chap dry ev...,3,1


In [ ]:
df_topik3.shape

(5312, 3)

In [ ]:
x=df_topik3['text']
y=df_topik3['sentiment']
print("Distribusi sentimen positif dan negatif dalam ulasan: \n{}".format(y.value_counts(normalize=True)))
# Split the dataset into test and train
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=50)

# As we saw above that data is imbalanced, balance training data using over sampling

ros = over_sampling.RandomOverSampler(random_state=0)
X_train, y_train = ros.fit_resample(pd.DataFrame(X_train), pd.Series(y_train))
print("Distribusi sentimen positif dan negatif dalam ulasan setelah oversampling: \n{}".format(y_train.value_counts(normalize=True)))
# Convert into list of string
X_train = X_train['text'].tolist()

Distribusi sentimen positif dan negatif dalam ulasan: 
sentiment
1    0.859375
0    0.140625
Name: proportion, dtype: float64
Distribusi sentimen positif dan negatif dalam ulasan setelah oversampling: 
sentiment
1    0.5
0    0.5
Name: proportion, dtype: float64


In [ ]:
word_vectorizer = TfidfVectorizer(strip_accents='unicode', token_pattern=r'\w{1,}',\
                                ngram_range=(1, 3), stop_words='english', sublinear_tf=True)

# Fiting it on Train
word_vectorizer.fit(X_train)
# Transforming the train and test datasets
X_train_transformed = word_vectorizer.transform(X_train)
X_test_transformed = word_vectorizer.transform(X_test.tolist())

In [ ]:
def evaluate_model(y_pred,y_actual):
    print(classification_report(y_true = y_actual, y_pred = y_pred))
    # Confusion matrix
    cm = confusion_matrix(y_true = y_actual, y_pred = y_pred)
    TN = cm[0, 0]
    FP = cm[0, 1]
    FN = cm[1, 0]
    TP = cm[1, 1]
    # Calculating the Sensitivity
    sensitivity = round(TP/float(FN + TP),2)
    print("sensitivity: {}".format(sensitivity))
    # Calculating the Specificity
    specificity = round(TN / float(TN + FP),2)
    print("specificity: {}".format(specificity))

#4.1 Model Training
logit = LogisticRegression()
logit.fit(X_train_transformed,y_train)
#4.2 Prediction on Train Data
y_pred_train= logit.predict(X_train_transformed)
#4.3 Prediction on Test Data
y_pred_test = logit.predict(X_test_transformed)
#4.4 Evaluation on Train
print("Evaluation on Train dataset ..")
evaluate_model(y_pred = y_pred_train, y_actual = y_train)
print("Evaluation on Test dataset ..")
#4.5 Evaluation on Test
evaluate_model(y_pred = y_pred_test, y_actual = y_test)

Evaluation on Train dataset ..
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      3192
           1       1.00      0.99      0.99      3192

    accuracy                           0.99      6384
   macro avg       0.99      0.99      0.99      6384
weighted avg       0.99      0.99      0.99      6384

sensitivity: 0.99
specificity: 1.0
Evaluation on Test dataset ..
              precision    recall  f1-score   support

           0       0.52      0.34      0.41       221
           1       0.90      0.95      0.92      1373

    accuracy                           0.86      1594
   macro avg       0.71      0.65      0.67      1594
weighted avg       0.85      0.86      0.85      1594

sensitivity: 0.95
specificity: 0.34


###Error Analysis

In [ ]:
true_labels = y_test.tolist()
pred_labels = y_pred_test.tolist()
texts = X_test.tolist()

error_df3 = pd.DataFrame({
    'text': texts,
    'actual': true_labels,
    'predicted': pred_labels
})

In [ ]:
# Inspect misclassified samples
misclassified_3 = error_df3[error_df3['actual'] != error_df3['predicted']]

print("Jumlah total error:", len(misclassified_3))
display(misclassified_3.sample(10, random_state=42))

Jumlah total error: 216


,text,actual,predicted
1481,please bring back cocoa lip shimmer havent wea...,1,0
1579,exceptional product shopping timely,0,1
982,terrible dry night time cough lately spring al...,0,1
1291,cool movie good movie little disappointed effe...,0,1
110,recommend mom rave movie wouldnt see hadnt rec...,0,1
821,great product lemon scent doesnt leave bleach ...,0,1
1297,discontinue im disappointed discontinue lip sh...,0,1
516,tasteless pay let check receipt blan,0,1
1496,chester excellent snack young child old also ...,1,0
983,cooky love cooky,1,0


###Sample df3

In [ ]:
sampled_df3 = error_df3.sample(n=300, random_state=42)
sampled_df3

,text,actual,predicted
917,exxtra gel nexxus exxtra gel long product buy ...,1,0
1120,touch gray black gray product easy use work co...,1,1
1276,super please find american grown black rice re...,1,1
361,something mary greatsaturday morning movie wak...,0,0
1483,magic lip love vaseline cocoa butter lip alway...,1,1
...,...,...,...
1564,amaze expect movie well first one certainly de...,1,1
1403,great love lip shimmer dont wear anything else...,1,1
495,poligrip last day good flavor,1,1
1383,lundberg wehani whole grain brown rice great p...,1,1


In [ ]:
# Inspect misclassified samples
misclassified_sample3 = sampled_df3[sampled_df3['actual'] != sampled_df3['predicted']]

print("Jumlah total error:", len(misclassified_sample3))
display(misclassified_sample3.sample(10, random_state=42))

misclassified_3.to_csv('misclassified3.csv', index=False)

Jumlah total error: 39


,text,actual,predicted
767,good movie daughter love christmas gift watch ...,0,1
163,great expect junk movie rock pick,0,1
1005,godzilla godzilla need say standard godzilla a...,1,0
162,kendrick plaza hysterical anna kendrick aubrey...,0,1
1083,god daughter love loed,0,1
339,bore story line nearly exciting plane child di...,0,1
23,little wipe big problem love wipe much efficie...,1,0
199,lifesaver cruise end sunburnt lips friend trav...,0,1
438,little slow start although little slow start a...,1,0
705,worth money worth every penny especially use b...,0,1


In [ ]:
sampled_with_acc_3 = sampled_df3.copy()
sampled_with_acc_3['acc_e'] = sampled_with_acc_3['text'].apply(
    lambda x: 0 if x in misclassified_sample3['text'].values else 1
)
display(sampled_with_acc_3)

,text,actual,predicted,acc_e
917,exxtra gel nexxus exxtra gel long product buy ...,1,0,0
1120,touch gray black gray product easy use work co...,1,1,1
1276,super please find american grown black rice re...,1,1,1
361,something mary greatsaturday morning movie wak...,0,0,1
1483,magic lip love vaseline cocoa butter lip alway...,1,1,1
...,...,...,...,...
1564,amaze expect movie well first one certainly de...,1,1,1
1403,great love lip shimmer dont wear anything else...,1,1,1
495,poligrip last day good flavor,1,1,1
1383,lundberg wehani whole grain brown rice great p...,1,1,1


In [ ]:
sampled_with_acc_3.to_csv('topik3_sampel.csv', index=False)